In [ ]:
import os
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ==========================================
# 1. ARCHITECTURE IMPORT
# ==========================================
# IMPORTANT: PyTorch needs to know what the model architecture looks like 
# before it can load the weights. 
# Either paste your entire RobustLungSegNet class (and its dependencies) here, 
# or import it if you saved it in a separate file like this:
# from model_architecture import RobustLungSegNet

# ==========================================
# 2. PREPROCESSING PIPELINE
# ==========================================
def window_and_normalize(image, wl=-600, ww=1500):
    """Applies the exact same CT Lung Window used during training."""
    hu_min = wl - (ww / 2)
    hu_max = wl + (ww / 2)
    image = np.clip(image, hu_min, hu_max)
    return (image - hu_min) / (hu_max - hu_min)

def load_and_preprocess(image_path, target_size=(256, 256)):
    """Loads an image or .npy file and preps it for the neural network."""
    ext = os.path.splitext(image_path)[1].lower()
    
    if ext == '.npy':
        img = np.load(image_path)
    else:
        # Load standard images (png, jpg) in grayscale
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not load image at {image_path}")
            
    # Resize to the network's expected input size
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    
    # Apply windowing if it's a raw CT array, or just normalize if it's already a standard image
    if img.max() > 255 or img.min() < 0:
        img = window_and_normalize(img)
    else:
        img = img.astype(np.float32) / 255.0
        
    # Convert to PyTorch Tensor: Add Batch and Channel dimensions -> Shape: (1, 1, 256, 256)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()
    
    return img_tensor, img

# ==========================================
# 3. INFERENCE & VISUALIZATION
# ==========================================
def run_clinical_inference(model_path, image_path, mc_passes=10):
    # Setup Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on: {device}")
    
    # Initialize Model & Load Weights
    print("Loading model weights...")
    model = RobustLungSegNet(num_classes=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    
    # Load and Preprocess Image
    print(f"Processing image: {image_path}")
    img_tensor, original_img_display = load_and_preprocess(image_path)
    img_tensor = img_tensor.to(device)
    
    # Run Monte Carlo Inference
    print(f"Running {mc_passes} Monte Carlo passes for Uncertainty Estimation...")
    mean_probs, uncertainty_map = model.predict_with_uncertainty(img_tensor, num_passes=mc_passes)
    
    # Process outputs
    prediction_mask = torch.argmax(mean_probs, dim=1).squeeze().cpu().numpy()
    uncertainty_map = uncertainty_map.squeeze().cpu().numpy()
    
    # Define Class Colors (0: Black, 1: Green, 2: Red, 3: Yellow)
    colors = np.array([
        [0, 0, 0],         
        [0, 200, 0],       
        [220, 50, 50],     
        [255, 220, 0]      
    ], dtype=np.uint8)
    
    pred_color = colors[prediction_mask]
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Original
    axes[0].imshow(original_img_display, cmap='gray')
    axes[0].set_title("Original CT Slice", fontsize=14)
    axes[0].axis('off')
    
    # 2. Prediction
    axes[1].imshow(pred_color)
    axes[1].set_title("AI Segmentation Mask", fontsize=14)
    axes[1].axis('off')
    
    # 3. Uncertainty
    im = axes[2].imshow(uncertainty_map, cmap='magma')
    axes[2].set_title("Uncertainty Heatmap", fontsize=14)
    axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    # Point these to your local files!
    MODEL_WEIGHTS = "best_robust_lung_model.pth"
    TEST_IMAGE = "path/to/your/test_image.png" # Can be .png, .jpg, or .npy
    
    run_clinical_inference(MODEL_WEIGHTS, TEST_IMAGE)